In [1]:
import numpy as np
import pandas as pd

from src.data_preparation import (
    load_data,
    prepare_data,
    FEATURES
)

df = load_data()
X_train, X_test, y_train, y_test = prepare_data(df)

print("Data loaded successfully.")

Data loaded successfully.


In [4]:
import joblib

rf_model = joblib.load(
    r"C:\docx\Predictive_Maintenance\best_random_forest.joblib"
)

anomaly_model = joblib.load(
    r"C:\docx\Predictive_Maintenance\isolation_forest.joblib"
)

print("Random Forest:", type(rf_model).__name__)
print("Isolation Forest:", type(anomaly_model).__name__)

Random Forest: Pipeline
Isolation Forest: Pipeline


In [5]:
print("Random Forest classes:", rf_model.classes_)
print("Random Forest fitted:", hasattr(rf_model, "predict_proba"))
print("Isolation Forest fitted:", hasattr(anomaly_model, "decision_function"))

Random Forest classes: [0 1]
Random Forest fitted: True
Isolation Forest fitted: True


## anomaly-score threshold

In [6]:
train_anomaly_scores = -anomaly_model.decision_function(X_train)

anomaly_threshold = np.percentile(
    train_anomaly_scores, 95
)

print("Anomaly threshold:", anomaly_threshold)

Anomaly threshold: 0.08741139806396735


In [10]:
def maintenance_decision(
    X,
    rf_model,
    anomaly_model,
    anomaly_threshold,
    low_risk=0.10,
    high_risk=0.50
):
    # 1. Random Forest failure probability
    classes = list(rf_model.classes_)
    failure_class_index = classes.index(1)

    failure_probability = rf_model.predict_proba(X)[
        :, failure_class_index
    ]

    # 2. Isolation Forest anomaly score
    anomaly_score = -anomaly_model.decision_function(X)

    anomaly_flag = anomaly_score >= anomaly_threshold

    # 3. Combine outputs into an initial priority rule
    priority = []

    for risk, is_anomaly in zip(
        failure_probability, anomaly_flag
    ):
        if risk >= high_risk and is_anomaly:
            priority.append("Critical")
        elif risk >= high_risk or (
            risk >= low_risk and is_anomaly
        ):
            priority.append("High")
        elif risk >= low_risk or is_anomaly:
            priority.append("Medium")
        else:
            priority.append("Low")

    result = pd.DataFrame(
        {
            "Failure Probability": failure_probability,
            "Anomaly Score": anomaly_score,
            "Anomaly Detected": anomaly_flag,
            "Maintenance Priority": priority,
        },
        index=X.index
    )

    result["Maintenance Needed"] = (
        result["Maintenance Priority"] != "Low"
    )

    return result

In [11]:
maintenance_results = maintenance_decision(
    X=X_test,
    rf_model=rf_model,
    anomaly_model=anomaly_model,
    anomaly_threshold=anomaly_threshold
)

display(maintenance_results.head(10))

print("\nMaintenance priority counts:")
print(
    maintenance_results["Maintenance Priority"]
    .value_counts()
)

print("\nMaintenance needed:")
print(
    maintenance_results["Maintenance Needed"]
    .value_counts()
)

,Failure Probability,Anomaly Score,Anomaly Detected,Maintenance Priority,Maintenance Needed
2997,0.296925,-0.017708,False,Medium,True
4871,0.000000,-0.025704,False,Low,False
3858,0.131073,-0.047933,False,Medium,True
951,0.000000,0.117857,True,Medium,True
6463,0.014828,0.078739,False,Low,False
3264,0.000000,0.061009,False,Low,False
4508,0.000000,-0.075671,False,Low,False
2100,0.000000,-0.035131,False,Low,False
7885,0.000000,-0.054453,False,Low,False
2421,0.034087,0.122748,True,Medium,True



Maintenance priority counts:
Maintenance Priority
Low         1747
Medium       180
High          62
Critical      11
Name: count, dtype: int64

Maintenance needed:
Maintenance Needed
False    1747
True      253
Name: count, dtype: int64


In [12]:
# Compare maintenance priorities with actual machine failures

validation = maintenance_results.copy()
validation["Actual Failure"] = y_test

print("Actual failures by maintenance priority:")
display(
    pd.crosstab(
        validation["Maintenance Priority"],
        validation["Actual Failure"],
        margins=True
    )
)

print("\nFailure rate within each priority:")
display(
    validation.groupby("Maintenance Priority")["Actual Failure"]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "count": "Number of Records",
        "sum": "Actual Failures",
        "mean": "Failure Rate"
    })
    .sort_values("Failure Rate", ascending=False)
    .assign(
        Failure_Rate_Percent=lambda x: (x["Failure Rate"] * 100).round(2)
    )
)

Actual failures by maintenance priority:


Actual Failure,0,1,All
Maintenance Priority,,,
Critical,1,10,11
High,20,42,62
Low,1742,5,1747
Medium,169,11,180
All,1932,68,2000



Failure rate within each priority:


,Number of Records,Actual Failures,Failure Rate,Failure_Rate_Percent
Maintenance Priority,,,,
Critical,11,10,0.909091,90.91
High,62,42,0.677419,67.74
Medium,180,11,0.061111,6.11
Low,1747,5,0.002862,0.29
